# Gradio UI with Toggle Architecture - Q&A System V2

Refactored to support both current UI and new design with toggle mechanism.

In [ ]:
%load_ext autoreload
%autoreload 2

# Now any changes to your modules will be automatically reloaded

In [ ]:
# Setup - adjust paths as needed
import sys
sys.path.append('..')


import gradio as gr
from qanda_module.config import setup_system
from qanda_module.retrieval_lean import process_question_with_style
import logging
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Enable logging to see what's happening
logging.basicConfig(level=logging.INFO)

print("📦 Imports successful!")

In [ ]:
import importlib
import qanda_module.retrieval_lean

# Force reload the module
importlib.reload(qanda_module.retrieval_lean)

In [ ]:
# Setup system - this should work from your previous testing
print("🚀 Setting up system...")

helpers, qa_chain, config = setup_system("../config.yaml")

print("✅ System setup complete!")
print(f"Model: {config.chat_model_name}")
print(f"Temperature: {config.temperature}")
print(f"Debug: {config.debug_enabled}")

In [25]:
# Curated popular topics for general use
POPULAR_TOPICS = [
    "Political Commentary", "Climate Change", "Economy", "Healthcare", "Education", "Housing", 
    "Immigration", "Energy", "Defence", "Employment", "Tax", 
    "Environment", "Trade"
]

# Full substantive topics for semantic matching
ALL_SUBSTANTIVE_TOPICS = [
    "Political Commentary", "Climate Change", "Economy", "Healthcare", "Education", "Housing", "Immigration", 
    "Energy", "Defence", "Transport", "Tax", "Welfare", "Mining", "Agriculture",
    "Technology", "Media", "Arts", "Indigenous", "Mental Health", "Aged Care",
    "Childcare", "Employment", "Industrial Relations", "Trade", "Foreign Policy",
    "Security", "Environment", "Water", "Regional Development", "Infrastructure",
    "Small Business", "Innovation", "Tourism", "Disability", "Youth", "Women",
    "Multiculturalism", "Refugees", "Border Protection", "Asylum Seekers", "Coal"
]

In [ ]:
# Create embedder in global scope for testing
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(config.embedding_model_name)
print("✅ Embedder created successfully")

In [ ]:
# Shared helper functions used by both UIs
def handle_question(question, k_value, style):
    """Handle user question and return response."""
    if not question.strip():
        return "Please enter a question.", "", "⚠️ No question provided"
    
    print(f"Processing: {question[:50]}...")  # Debug print
    
    try:
        result = process_question_with_style(
            helpers=helpers,
            qa_chain=qa_chain, 
            query=question,
            k=k_value,
            style=style,
            config=config
        )
        
        # result = (formatted_answer, sources, status, pdf_output, tech_info)
        return result[0], result[1], result[2]
        
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(f"❌ Error in handler: {error_msg}")
        return error_msg, "", f"❌ {error_msg}"

def generate_sample_questions(panelist=None, topic=None):
    """Generate comprehensive sample questions."""
    questions = []
    
    # Specific combinations first
    if panelist and topic:
        questions.extend([
            f"What are {panelist}'s views on {topic.lower()}?",
            f"How did {panelist} respond to questions about {topic.lower()}?",
        ])
    
    # Just panelist questions
    if panelist:
        questions.extend([
            f"What did {panelist} say about climate change?",
            f"What are {panelist}'s most controversial views?",
            f"How did {panelist} handle tough questions?",
        ])
    
    # Just topic questions  
    if topic:
        questions.extend([
            f"What did panelists say about {topic.lower()}?",
            f"What are different perspectives on {topic.lower()}?",
            f"Which panelists had the strongest views on {topic.lower()}?",
        ])
    
    # Always include general questions
    questions.extend([
        "What are the most controversial topics discussed?",
        "Which panelists had the most heated exchanges?",
        "What are the main political divides shown?",
        "What topics generated the most debate?",
    ])
    
    # Remove duplicates while preserving order
    seen = set()
    unique_questions = []
    for q in questions:
        if q not in seen:
            seen.add(q)
            unique_questions.append(q)
    
    return unique_questions[:10]  # Limit to 10 questions

def update_sample_questions(panelist, topic):
    new_questions = generate_sample_questions(panelist, topic)
    return gr.update(choices=new_questions, value=None)

print("✅ Shared helper functions defined")

In [30]:
def get_latest_panelist_data(con):
    """Get latest profession and URL for each panelist"""
    
    panelist_data = con.execute("""
        SELECT name, profession, link,
               ROW_NUMBER() OVER (PARTITION BY name ORDER BY id DESC) as rn
        FROM dim_panellist 
        WHERE name IS NOT NULL
    """).df()
    
    # Keep only latest entry for each panelist
    latest_panelists = panelist_data[panelist_data['rn'] == 1]
    
    # Create lookup dictionary: name -> (profession, link)
    panelist_lookup = {}
    for _, row in latest_panelists.iterrows():
        panelist_lookup[row['name']] = (row['profession'], row['link'])
    
    return panelist_lookup

def format_selection_display(panelist="", topic=""):
    """Format current selection display with clickable panelist profiles"""
    
    if not panelist and not topic:
        return "**Current Focus:** None selected"
    
    parts = []
    
    # Format panelist with profile link if available
    if panelist:
        # Case-insensitive lookup - find matching key
        matching_key = None
        for key in panelist_lookup.keys():
            if key.upper() == panelist.upper():
                matching_key = key
                break
        
        if matching_key:
            profession, url = panelist_lookup[matching_key]
            profession_text = profession if profession and str(profession) != 'nan' else 'Panelist'
            
            if url and str(url) != 'nan':
                panelist_part = f"👤 [{matching_key}]({url}) *({profession_text})*"
            else:
                panelist_part = f"👤 {matching_key} *({profession_text})*"
        else:
            panelist_part = f"👤 {panelist}"
        
        parts.append(panelist_part)
    
    # Add topic if selected
    if topic:
        parts.append(f"🎯 {topic}")
    
    return "**Current Focus:** " + " + ".join(parts)

# Initialize the lookup once
import duckdb
con = duckdb.connect(config.duck_db_name)
panelist_lookup = get_latest_panelist_data(con)
print(f"Loaded {len(panelist_lookup)} panelists with profile data")

globals()['panelist_lookup'] = panelist_lookup
# Add this line right after creating panelist_lookup:
helpers.panelist_lookup = panelist_lookup
helpers.con = con  
print("Added panelist_lookup to helpers object")

Loaded 1094 panelists with profile data
Added panelist_lookup to helpers object


In [31]:
# Quick test
print("helpers has con:", hasattr(helpers, 'con'))
print("helpers has panelist_lookup:", hasattr(helpers, 'panelist_lookup'))
if hasattr(helpers, 'panelist_lookup'):
    print("panelist_lookup size:", len(helpers.panelist_lookup))

helpers has con: True
helpers has panelist_lookup: True
panelist_lookup size: 1094


In [32]:
test_name = "CHRISTOPHER PYNE"
if test_name in panelist_lookup:
    profession, url = panelist_lookup[test_name]
    print(f"Test: {test_name} -> Profession: {profession}, URL: {url}")

In [33]:
test_name.title()

'Christopher Pyne'

In [34]:
panelist_lookup

{'Alexander Downer': ('former Foreign Affairs Minister',
  'https://www.abc.net.au/qanda/alexander-downer/10644430'),
 'Alissa Nutting': ('American author and academic.',
  'https://www.abc.net.au/qanda/alissa-nutting/10642586'),
 'Anna Greenberg': ('Pollster and Commentator',
  'https://www.abc.net.au/qanda/anna-greenberg/10641322'),
 'Bob Cronin': ('Former Leader of the Greens',
  'https://www.abc.net.au/qanda/bob-cronin/10643130'),
 'Brett Solomon': ('Executive Director of Accessnow',
  'https://www.abc.net.au/qanda/brett-solomon/10643882'),
 'Bruce Billson': ('Liberal Party MP',
  'https://www.abc.net.au/qanda/bruce-billson/10641556'),
 'Bruce Guthrie': ('former News Ltd editor and author of Man Bites Murdoch',
  'https://www.abc.net.au/qanda/bruce-guthrie/10644090'),
 'Bryan Stevenson': ('Director, Equal Justice Initiative',
  'https://www.abc.net.au/qanda/bryan-stevenson/10641926'),
 'Charles Waterstreet': ('flamboyant barrister',
  'https://www.abc.net.au/qanda/charles-waterstre

In [35]:
def build_panelist_text_profile(helpers, panelist_name):
    """Build rich text profile for semantic matching"""
    if not panelist_name:
        return ""
    
    # Get panelist's responses
    panelist_data = helpers.df_reply[
        (helpers.df_reply["speaker_name"] == panelist_name) & 
        (helpers.df_reply["speaker_type"] == 3)
    ]
    
    # Collect subtopics
    subtopics = panelist_data["subtopic"].dropna().unique()
    subtopic_text = " ".join(subtopics)
    
    # Get episode titles they appeared in
    episode_ids = panelist_data["episode_id"].unique()
    episodes = helpers.df_ep[helpers.df_ep["id"].isin(episode_ids)]
    episode_titles = " ".join(episodes["title"].fillna(""))
    
    # Combine for rich profile
    profile = f"{subtopic_text} {episode_titles}".strip()
    return profile if profile else "general political discussion"


def get_semantic_topic_matches(helpers, panelist_name, embedder, limit=8, threshold=0.3):
    """Get semantically similar topics for panelist - with better error handling"""
    if not panelist_name:
        return POPULAR_TOPICS
    
    try:
        # Build panelist profile
        profile = build_panelist_text_profile(helpers, panelist_name)
        
        # Check for empty profile
        if not profile or len(profile.strip()) < 5:
            print(f"Warning: Empty profile for {panelist_name}, using popular topics")
            return POPULAR_TOPICS
        
        # Get embeddings
        profile_embedding = embedder.encode([profile])
        topic_embeddings = embedder.encode(ALL_SUBSTANTIVE_TOPICS)
        
        # Check for valid embeddings
        if profile_embedding.shape[1] == 0 or topic_embeddings.shape[1] == 0:
            return POPULAR_TOPICS
        
        # Calculate similarities with error handling
        from sklearn.metrics.pairwise import cosine_similarity
        import numpy as np
        
        similarities = cosine_similarity(profile_embedding, topic_embeddings)[0]
        
        # Filter out NaN/inf values
        valid_similarities = [(topic, score) for topic, score in zip(ALL_SUBSTANTIVE_TOPICS, similarities) 
                             if not (np.isnan(score) or np.isinf(score)) and score > threshold]
        
        valid_similarities.sort(key=lambda x: x[1], reverse=True)
        matched_topics = [topic for topic, _ in valid_similarities[:limit]]
        
        if len(matched_topics) < 4:
            return POPULAR_TOPICS
        
        return matched_topics
        
    except Exception as e:
        print(f"Semantic matching failed for {panelist_name}: {e}")
        return POPULAR_TOPICS

In [36]:
def build_panelist_search_index(helpers):
    """Build searchable index of all panelists with metadata"""
    df_responses = helpers.df_reply
    df_panelists = helpers.df_guests
    
    # Get panelist stats from responses  
    panelist_stats = df_responses[df_responses["speaker_type"] == 3].groupby("speaker_name").agg({
        "episode_id": "nunique",  # Number of episodes appeared in
        "id": "count"            # Total number of responses
    }).rename(columns={"episode_id": "episodes", "id": "responses"})
    
    # Build search index
    search_index = []
    for name in panelist_stats.index:
        if pd.isna(name) or not name.strip():
            continue
            
        stats = panelist_stats.loc[name]
        
        # Try to find profile URL
        profile_url = None
        matching_panelist = df_panelists[df_panelists["name"] == name]
        if not matching_panelist.empty and "link" in df_panelists.columns:
            profile_url = matching_panelist.iloc[0]["link"]
            if pd.isna(profile_url):
                profile_url = None
        
        search_index.append({
            "name": name,
            "episodes": int(stats["episodes"]),
            "responses": int(stats["responses"]),
            "profile_url": profile_url,
            "search_text": name.lower()
        })
    
    # Sort by total responses (most active first)
    search_index.sort(key=lambda x: x["responses"], reverse=True)
    return search_index

def search_panelists(query, search_index, limit=8):
    """Fuzzy search panelists"""
    if not query or len(query) < 2:
        return []
    
    query_lower = query.lower()
    matches = []
    
    for panelist in search_index:
        name = panelist["name"]
        search_text = panelist["search_text"]
        
        # Exact match gets highest score
        if query_lower == search_text:
            score = 100
        # Starts with query gets high score  
        elif search_text.startswith(query_lower):
            score = 90
        # Contains query gets medium score
        elif query_lower in search_text:
            score = 70
        # Fuzzy match on words
        elif any(word.startswith(query_lower) for word in search_text.split()):
            score = 60
        else:
            continue
        
        # Boost score by activity level
        activity_boost = min(panelist["responses"] / 100, 10)
        final_score = score + activity_boost
        
        matches.append((final_score, panelist))
    
    # Sort by score and return top matches
    matches.sort(key=lambda x: x[0], reverse=True)
    return [match[1] for match in matches[:limit]]

def get_popular_panelists(search_index, limit=6):
    """Get most active panelists"""
    return search_index[:limit]

print("✅ Panelist search functions defined")

✅ Panelist search functions defined


In [37]:
def generate_smart_questions(panelist="", topic=""):
    """Generate questions based on selections - updated with better topic-only templates"""
    if panelist and topic:
        return [
            f"What are {panelist}'s views on {topic.lower()}?", 
            f"How did {panelist} handle {topic.lower()} questions?",
            f"What's {panelist}'s stance on {topic.lower()} policy?"
        ]
    elif topic == "General Politics":
        if panelist:
            return [
                f"What are {panelist}'s most notable political views?",
                f"How has {panelist}'s political stance evolved over time?",
                f"What are {panelist}'s most controversial political positions?"
            ]
        else:
            return [
                "What are the most controversial political debates?",
                "How have political views changed over the years?",
                "What are the biggest political divisions in Australia?"
            ]
    elif panelist:
        return [
            f"What are {panelist}'s most notable views?",
            f"Has {panelist} changed their political views over time?",
            f"What are {panelist}'s most controversial positions?",
            f"How did {panelist} handle tough questions?"
        ]
    elif topic:
        # NEW: Better topic-only questions for general exploration
        return [
            f"What do panelists think of {topic.lower()}?",
            f"What are different perspectives on {topic.lower()}?",
            f"Which panelists have strong views on {topic.lower()}?",
            f"How do Labor and Liberal differ on {topic.lower()}?",
            f"What are the most controversial views on {topic.lower()}?",
            f"How have panelist views on {topic.lower()} evolved?"
        ]
    else:
        return [
            "What do panelists think of climate change?",
            "How do panelists view immigration policy?", 
            "What are different perspectives on the economy?",
            "Which panelists had the most controversial views?"
            "What are the most controversial political debates?",
            "How have political views changed over the years?",
            "What are the biggest political divisions in Australia?"            
        ]

def format_selection_display(panelist="", topic=""):
    if not panelist and not topic:
        return "**Current Focus:** None selected"
    
    parts = []
    if panelist:
        # Find matching panelist (case-insensitive)
        matching_key = None
        for key in panelist_lookup.keys():
            if key.upper() == panelist.upper():
                matching_key = key
                break
        
        if matching_key:
            profession, url = panelist_lookup[matching_key]
            profession_text = profession if profession and str(profession) != 'nan' else 'Panelist'
            if url and str(url) != 'nan':
                panelist_part = f"👤 [{matching_key}]({url}) *({profession_text})*"
            else:
                panelist_part = f"👤 {matching_key} *({profession_text})*"
        else:
            panelist_part = f"👤 {panelist}"
        parts.append(panelist_part)
    
    if topic:
        parts.append(f"🎯 {topic}")
    
    return "**Current Focus:** " + " + ".join(parts)

def create_topic_buttons_grid(topics, per_row=2):
    """Create topic button grid - returns button list"""
    buttons = []
    for i in range(0, len(topics), per_row):
        with gr.Row():
            for j in range(per_row):
                if i + j < len(topics):
                    topic = topics[i + j]
                    btn = gr.Button(topic, variant="secondary", elem_classes=["topic-chip"])
                    buttons.append(btn)
    return buttons

def safe_build_panelist_list(helpers):
    """Safely build panelist list with fallback"""
    try:
        panelist_search_index = build_panelist_search_index(helpers)
        return [p["name"] for p in panelist_search_index[:100]]
    except Exception as e:
        print(f"Warning: Using fallback panelist list: {e}")
        return ["Malcolm Turnbull", "Penny Wong", "Tony Abbott", "Scott Morrison", "Anthony Albanese"]
    

# Add this debug function:
def debug_panelist_change(panelist_display):
    """Debug what happens when panelist changes"""
    clean_panelist = extract_panelist_name(panelist_display)
    print(f"🔍 Debug: Panelist = '{clean_panelist}'")
    
    if clean_panelist:
        relevant_topics = get_semantic_topic_matches(helpers, clean_panelist, embedder)
        print(f"🎯 Semantic topics: {relevant_topics}")
    else:
        print(f"🎯 No panelist - using popular topics: {POPULAR_TOPICS}")
    
    return clean_panelist


In [38]:
def enhanced_build_panelist_list(helpers):
    """Build panelist list sorted by episodes, with subtle appearance counts"""
    try:
        search_index = build_panelist_search_index(helpers)
        
        # Sort by episodes (most frequent guests first)
        search_index.sort(key=lambda x: x["episodes"], reverse=True)
        
        # Format: "Name (X eps)" - short and clean
        enhanced_list = [f"{p['name']} ({p['episodes']} eps)" for p in search_index[:100]]
        return enhanced_list
    except Exception as e:
        print(f"Warning: Using fallback: {e}")
        return ["Malcolm Turnbull (15 eps)", "Penny Wong (12 eps)"]

def extract_panelist_name(display_text):
    """Extract just the name from 'Name (X eps)' format"""
    if not display_text or display_text == "":
        return ""
    return display_text.split(" (")[0] if " (" in display_text else display_text

def get_panelist_episodes(helpers, panelist_name):
    """Get episodes for selected panelist - succinct version"""
    if not panelist_name:
        return "*Select a panelist to see their episodes*"
    
    try:
        # Use existing helpers - much simpler!
        episodes = helpers.df_reply[
            (helpers.df_reply["speaker_name"] == panelist_name) & 
            (helpers.df_reply["speaker_type"] == 3)
        ].merge(helpers.df_ep, left_on="episode_id", right_on="id")["ep_label"].unique()
        
        if len(episodes) == 0:
            return f"*No episodes found for {panelist_name}*"
        
        # Sort and format
        sorted_episodes = sorted(episodes, reverse=True)
        header = f"**{panelist_name} appeared in {len(episodes)} episodes:**\n\n"
        episode_list = "\n".join([f"• {ep}" for ep in sorted_episodes])
        
        return header + episode_list
    except Exception as e:
        return f"*Error loading episodes: {str(e)}*"

print("✅ Enhanced panelist functions defined")

✅ Enhanced panelist functions defined


In [39]:
def wire_ui_events(panelist_dropdown, topic_radio, topic_label, sample_questions, question,  # Added topic_label
                   ask_btn, clear_btn, back_btn, clear_question_btn, current_panelist, 
                   current_topic, current_selection, status_display, answer_display, 
                   sources_display, tabs, k_slider, style_radio, episodes_display):
    """Wire up all UI events - FIXED with proper indentation"""
    
    def update_selections(panelist, topic):
        questions = generate_smart_questions(panelist, topic)
        display = format_selection_display(panelist, topic)
        selected = questions[0] if (panelist or topic) else None
        status = f"Generated {len(questions)} questions" if (panelist or topic) else "Ready"
        return display, gr.update(choices=questions, value=selected), selected or "", status
     
    def on_panelist_change(panelist_display, topic):
        clean_panelist = extract_panelist_name(panelist_display)
        episodes_text = get_panelist_episodes(helpers, clean_panelist)
        
        if clean_panelist:
            relevant_topics = get_semantic_topic_matches(helpers, clean_panelist, embedder)
            topic_label_text = f"**{clean_panelist}'s Topics** ({len(relevant_topics)} most relevant)"
            topic_choices = relevant_topics
        else:
            topic_label_text = "**Popular Topics**"
            topic_choices = POPULAR_TOPICS
        
        # Get the update_selections results
        selection_results = update_selections(clean_panelist, topic)
        
        return (
            clean_panelist,                                    # current_panelist (State)
            episodes_text,                                     # episodes_display (Textbox)  
            topic_label_text,                                  # topic_label (Markdown)
            gr.update(choices=topic_choices, value=None),      # topic_radio (Radio)
            selection_results[0],                              # current_selection (Markdown)
            selection_results[1],                              # sample_questions (Radio) 
            selection_results[2],                              # question (Textbox)
            selection_results[3]                               # status_display (Textbox)
        )
    
    def on_topic_click(topic_name, panelist, current_topic_value):
        new_topic = "" if current_topic_value == topic_name else topic_name
        return (new_topic,) + update_selections(panelist, new_topic)
    
    def clear_all():
        return "", "*Select a panelist to see their episodes*", "**Popular Topics**", gr.update(choices=POPULAR_TOPICS, value=None), "", format_selection_display(), gr.update(choices=generate_smart_questions(), value=None), "", "Cleared"

    def clear_question_only():
        return ""
        
    def ask_question(q, k, s):
        """FIXED: Better error handling and correct function call"""
        if not q.strip():
            return "**Please enter a question**", "", "⚠️ No question provided", gr.update()
        
        print(f"🔍 Processing question: {q}")  # Debug print
        print(f"🔍 Parameters: k={k}, style={s}")  # Debug print
        
        try:
            # FIXED: Use the original handle_question signature (it expects global helpers, qa_chain, config)
            result = handle_question(q, k, s)
            print(f"✅ Question processed successfully")  # Debug print
            return result[0], result[1], result[2], gr.update(selected=1)
            
        except Exception as e:
            # BETTER ERROR LOGGING
            import traceback
            error_details = traceback.format_exc()
            print(f"❌ Error in ask_question: {str(e)}")
            print(f"❌ Full traceback:\n{error_details}")
            
            return f"Error: {str(e)}", "", f"❌ Error: {str(e)}", gr.update()
    
    # Wire events (same as before)
    panelist_dropdown.change(
        fn=on_panelist_change,
        inputs=[panelist_dropdown, current_topic],
        outputs=[current_panelist, episodes_display, topic_label, topic_radio, current_selection, sample_questions, question, status_display]
    )
    
    # Update topic click handler:
    topic_radio.change(
        fn=lambda topic_name, panelist: (topic_name,) + update_selections(panelist, topic_name),
        inputs=[topic_radio, current_panelist],
        outputs=[current_topic, current_selection, sample_questions, question, status_display]
    )
    
    sample_questions.change(lambda q: q or "", [sample_questions], [question])
    clear_question_btn.click(clear_question_only, outputs=[question])
    clear_btn.click(clear_all, outputs=[current_panelist, episodes_display, current_topic, current_selection, sample_questions, question, status_display])
    ask_btn.click(ask_question, [question, k_slider, style_radio], [answer_display, sources_display, status_display, tabs])
    back_btn.click(lambda: gr.update(selected=0), outputs=[tabs])

In [40]:
def create_semantic_ui(helpers, qa_chain, config):
    """Clean, concise UI function using external helpers"""
    
    # Data setup
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(config.embedding_model_name)

    popular_topics = POPULAR_TOPICS[:8]
    more_topics = POPULAR_TOPICS[8:] if len(POPULAR_TOPICS) > 8 else []
    panelist_names = enhanced_build_panelist_list(helpers)
    
    with gr.Blocks(
        title="Q&A System V2", 
        theme=gr.themes.Soft(),
        css = """
        .topic-chip{margin:2px!important;padding:4px 8px!important;border-radius:12px!important;background:#f8fafc!important;border:1px solid #e2e8f0!important;transition:all 0.2s!important}
        .topic-chip:hover{background:#2563eb!important;color:white!important}
        #question-container{position:relative!important}
        #clear-btn-inside{position:absolute!important;top:32px!important;right:8px!important;width:20px!important;height:20px!important;min-width:20px!important;padding:0!important;font-size:12px!important;opacity:0.5!important;z-index:10!important;border-radius:50%!important}
        #clear-btn-inside:hover{opacity:0.8!important}
        """
    ) as demo:
        
        gr.Markdown("# 🚀 Q&A System V2")
        
        with gr.Tabs() as tabs:
            with gr.Tab("🔍 Ask Question", id=0):
                with gr.Row():
                    # LEFT COLUMN
                    with gr.Column(scale=35):
                        gr.Markdown("### 👤 Panelist")
                        panelist_dropdown = gr.Dropdown(
                            choices=[""] + panelist_names, 
                            value="", 
                            filterable=True, 
                            label="👤 Panelist",
                            info="Search by name (sorted by episode frequency)"
                        )
                        episodes_display = gr.Textbox(
                            value="*Select a panelist to see their episodes*", 
                            label="📅 All Episode Appearances",
                            lines=6,           # Show 6 lines
                            max_lines=6,       # Max 6 lines, then scroll
                            interactive=False,
                            elem_id="episodes-display"
                        )
                                                            
                        # WITH THIS:
                        with gr.Column() as topics_section:
                            topic_label = gr.Markdown("**Popular Topics**")
                            topic_radio = gr.Radio(
                                choices=POPULAR_TOPICS,
                                interactive=True,
                                label="",
                                elem_classes=["topic-radio"]
                            )
                        with gr.Row():
                            k_slider = gr.Slider(5, 50, 20, step=5, label="Docs")
                            style_radio = gr.Radio(["Concise", "Balanced", "Detailed"], value="Balanced", label="Style")
                                        
                    # RIGHT COLUMN
                    with gr.Column(scale=65):
                        current_selection = gr.Markdown("**Current Selection:** None")
                        gr.Markdown("### 💡 Sample Questions")
                        sample_questions = gr.Radio(choices=generate_smart_questions(), interactive=True, label="")

                        gr.Markdown("### 📝 Your Question")
                        with gr.Row():
                            with gr.Column(scale=1, elem_id="question-container"):
                                question = gr.Textbox("", placeholder="Enter your question...", lines=4, label="")
                                clear_question_btn = gr.Button("✕", variant="secondary", size="sm", elem_id="clear-btn-inside")
                        
                        # ADD THESE MISSING BUTTONS:
                        with gr.Row():
                            ask_btn = gr.Button("🔍 Ask Question", variant="primary", scale=3)
                            clear_btn = gr.Button("🗑️ Clear All", variant="secondary", scale=1)
                        
                        status_display = gr.Textbox("Ready", label="Status", interactive=False)            
            with gr.Tab("📝 Response", id=1):
                gr.Markdown("### 📝 Answer")
                answer_display = gr.Markdown("**Click 'Ask Question' to see response**")
                gr.Markdown("### 📚 Sources")
                sources_display = gr.Markdown("*Sources will appear here*")
                back_btn = gr.Button("← Back", variant="secondary")
        
        # State & Events (using external helpers)
        current_panelist = gr.State("")
        current_topic = gr.State("")
        
        # Wire up all events using clean external helper functions
        wire_ui_events(
            panelist_dropdown, topic_radio, topic_label, sample_questions, question, ask_btn, clear_btn, back_btn, clear_question_btn,  # Added topic_label
            current_panelist, current_topic, current_selection, status_display,
            answer_display, sources_display, tabs, k_slider, style_radio, episodes_display
        )
    
    return demo

In [41]:
def launch_ui_with_toggle(helpers, qa_chain, config, design="current"):
    """Main launcher with design selection"""
    
    print(f"🎨 Creating {design} UI...")
    
    if design == "new":
        demo = create_semantic_ui(helpers, qa_chain, config)  
        print("✅ New UI created")
    else:
        demo = create_current_ui(helpers, qa_chain, config)
        print("✅ Current UI created")
    
    return demo

print("✅ Toggle mechanism defined")

✅ Toggle mechanism defined


In [42]:
# Choose which UI to use
USE_NEW_DESIGN = True  # Set to True to test new UI placeholder

# Create the demo
demo = launch_ui_with_toggle(
    helpers, 
    qa_chain, 
    config, 
    design="new" if USE_NEW_DESIGN else "current"
)

print(f"🎯 Demo created using {'NEW' if USE_NEW_DESIGN else 'CURRENT'} design")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


🎨 Creating new UI...
✅ New UI created
🎯 Demo created using NEW design


In [43]:
# Launch the interface
print("🌐 Launching Gradio interface...")

demo.launch(
    share=True,  # Set to True for public link
    server_port=7860,
    show_error=True,
    debug=True  # Shows more info in console
)

🌐 Launching Gradio interface...


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Question processed successfully
🔍 Processing question: What are different perspectives on the economy?
🔍 Parameters: k=50, style=Balanced
Processing: What are different perspectives on the economy?...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


✅ Question processed successfully
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://58382420ec73ecbb3b.gradio.live


In [ ]:
import importlib
import qanda_module.retrieval_lean

# Force reload the module
importlib.reload(qanda_module.retrieval_lean)

print("Module reloaded successfully")

# Test immediately
from qanda_module.retrieval_lean import format_response_with_episode_links

# Quick test with a sample response
test_response = "Anthony Albanese argued about climate policy Episode ID: 394 and economic issues Episode ID: 300."
test_result = format_response_with_episode_links(test_response, con, helpers)
print("Test result:")
print(test_result)

In [44]:
# If you need to stop the demo
demo.close()

Closing server running on port: 7860


In [ ]:
# Import the function from the module
from qanda_module.retrieval_lean import format_response_with_episode_links

# First, get some documents and generate a response manually
query = "What are Malcolm Turnbull's views?"
docs = qa_chain.retriever._get_relevant_documents(query)
print(f"Retrieved {len(docs)} documents")

# Generate AI response
from qanda_module.retrieval_lean import generate_ai_response
ai_response = generate_ai_response(docs, query, "Balanced", config)
print(f"AI response generated, length: {len(ai_response)}")

# Now test our debug function directly
debug_result = format_response_with_episode_links(ai_response, con, helpers)

In [ ]:
# Run this in your notebook to verify helpers has panelist_lookup
print("Checking helpers object:")
print(f"helpers has panelist_lookup: {hasattr(helpers, 'panelist_lookup')}")
if hasattr(helpers, 'panelist_lookup'):
    print(f"panelist_lookup size: {len(helpers.panelist_lookup)}")
    # Check for Malcolm
    malcolm_found = [name for name in helpers.panelist_lookup.keys() if 'malcolm' in name.lower()]
    print(f"Malcolm entries: {malcolm_found}")

## 🧪 Quick Testing & Toggle Demo

Use these cells to test both UIs:

In [74]:
import duckdb
con = duckdb.connect("../data/qanda_v2.duckdb")  

In [ ]:
con.execute("SHOW TABLES").df()

In [76]:
df_guests = con.execute("SELECT * FROM dim_panellist LIMIT 5").df()


In [77]:
df_ep = con.execute("SELECT * FROM dim_episode LIMIT 5").df()


In [ ]:
pd.set_option('display.max_colwidth', 200)

In [ ]:
df_guests.head().T

In [ ]:
df_ep.head().T

In [ ]:
config

In [27]:
from chromadb import PersistentClient

In [ ]:
client = PersistentClient(path=config.chroma_path)
collection = client.get_or_create_collection(config.collection_name)

In [ ]:
# Quick ChromaDB metadata inspection
sample_docs = collection.get(limit=3, include=["metadatas", "documents"])

print("=== ChromaDB Metadata Check ===")
for i, metadata in enumerate(sample_docs["metadatas"]):
    print(f"\nDocument {i+1} metadata keys:")
    print(list(metadata.keys()))
    print(f"Sample metadata: {metadata}")
    
    # Check for episode_id specifically
    if "episode_id" in metadata:
        print(f"✅ episode_id found: {metadata['episode_id']}")
    else:
        print("❌ episode_id not found")

In [ ]:
# Test Current UI
print("Testing Current UI...")
current_demo = create_current_ui(helpers, qa_chain, config)
print("✅ Current UI created successfully")

# Uncomment to launch:
# current_demo.launch(server_port=7861, show_error=True)

In [ ]:
# Test New UI placeholder
print("Testing New UI placeholder...")
new_demo = create_new_ui(helpers, qa_chain, config)
print("✅ New UI placeholder created successfully")

# Uncomment to launch:
# new_demo.launch(server_port=7862, show_error=True)

In [ ]:
# Quick pipeline test (reusable)
def test_pipeline():
    test_question = "What did panelists say about climate change?"
    result = handle_question(test_question, 10, "Concise")
    print(f"✅ Pipeline test: {result[2]}")
    print(f"Answer length: {len(result[0])} chars")
    return result

# Uncomment to test:
# test_pipeline()

In [ ]:
# Hour 1: Topic Discovery Analysis - Fixed
print("🔍 Analyzing database for popular topics...")

# Get database connection
import duckdb
con = duckdb.connect(config.duck_db_name)

# Or alternatively, check if helpers has the connection
# print("Available in helpers:", [attr for attr in dir(helpers) if not attr.startswith('_')])

# Query most discussed topics by panelists
query = """
SELECT 
    subtopic,
    COUNT(*) as response_count,
    COUNT(DISTINCT episode_id) as episode_count,
    COUNT(DISTINCT speaker_name) as panelist_count
FROM fact_responses 
WHERE speaker_type = 3 
    AND subtopic IS NOT NULL 
    AND subtopic != ''
    AND LENGTH(TRIM(subtopic)) > 2
GROUP BY subtopic
ORDER BY response_count DESC
LIMIT 20
"""

topic_stats = con.execute(query).df()
print(f"✅ Found {len(topic_stats)} popular topics")

# Rest of the code stays the same...

In [ ]:
# Extended Topic Analysis - Top 50 for Manual Curation
print("🔍 Analyzing top 50 topics for manual filtering...")

# Get top 50 topics with better analysis
query = """
SELECT 
    subtopic,
    COUNT(*) as response_count,
    COUNT(DISTINCT episode_id) as episode_count,
    COUNT(DISTINCT speaker_name) as panelist_count,
    AVG(LENGTH(content)) as avg_response_length
FROM fact_responses 
WHERE speaker_type = 3 
    AND subtopic IS NOT NULL 
    AND subtopic != ''
    AND LENGTH(TRIM(subtopic)) > 2
GROUP BY subtopic
ORDER BY response_count DESC
LIMIT 50
"""

topic_stats = con.execute(query).df()
print(f"✅ Found {len(topic_stats)} topics")

print("\n🔥 Top 50 Topics for Review:")
print("=" * 90)
for i, row in topic_stats.iterrows():
    # Add markers to help identify good vs procedural topics
    marker = ""
    topic = row['subtopic']
    
    # Flag obviously procedural topics
    if any(x in topic.upper() for x in ['QUESTION', 'FLOOR', 'FOLLOW UP', 'BATMAN', 'JOKER']):
        marker = "🚫"
    # Flag potentially good topics
    elif any(x in topic.upper() for x in ['CLIMATE', 'ECONOMY', 'HEALTH', 'EDUCATION', 'HOUSING', 'IMMIGRATION', 'ASYLUM']):
        marker = "✅"
    # Flag specific events/people (might be too specific)
    elif any(x in topic.upper() for x in ['GONSKI', 'MALAYSIAN', 'INDONESIAN', 'BATMAN']):
        marker = "⚠️"
    else:
        marker = "🤔"
    
    print(f"{i+1:2d}. {marker} {topic:35} | {row['response_count']:3d} resp | {row['episode_count']:2d} eps | {row['panelist_count']:2d} people")

print("\n" + "=" * 90)
print("Legend:")
print("✅ = Likely good broad topic")
print("🚫 = Procedural/administrative") 
print("⚠️ = Specific event/policy (might be too narrow)")
print("🤔 = Needs review")

print(f"\n📊 Stats:")
print(f"Total topics: {len(topic_stats)}")
procedural = len([t for t in topic_stats['subtopic'] if any(x in t.upper() for x in ['QUESTION', 'FLOOR', 'FOLLOW'])])
print(f"Procedural topics: {procedural}")
print(f"Substantive topics: {len(topic_stats) - procedural}")

In [ ]:
# Filter and Display Substantive Topics
print("🔍 Filtering for substantive topics only...")

# Define filters
procedural_keywords = ['QUESTION', 'FLOOR', 'FOLLOW UP', 'BATMAN', 'JOKER', 'ABC', 'FROM THE']
specific_event_keywords = ['GONSKI', 'MALAYSIAN', 'INDONESIAN', 'ONE NATION', 'NBN']

substantive_topics = []
specific_events = []
procedural_topics = []

for i, row in topic_stats.iterrows():
    topic = row['subtopic']
    topic_upper = topic.upper()
    
    # Categorize topics
    if any(keyword in topic_upper for keyword in procedural_keywords):
        procedural_topics.append(row)
    elif any(keyword in topic_upper for keyword in specific_event_keywords):
        specific_events.append(row)
    else:
        substantive_topics.append(row)

print(f"\n✅ SUBSTANTIVE TOPICS ({len(substantive_topics)}):")
print("=" * 70)
for i, row in enumerate(substantive_topics):
    print(f"{i+1:2d}. {row['subtopic']:35} | {row['response_count']:3d} resp | {row['episode_count']:2d} eps")

print(f"\n⚠️  SPECIFIC EVENTS/POLICIES ({len(specific_events)}) - might be useful:")
for i, row in enumerate(specific_events[:10]):  # Show top 10
    print(f"{i+1:2d}. {row['subtopic']:35} | {row['response_count']:3d} resp | {row['episode_count']:2d} eps")

print(f"\n🚫 PROCEDURAL ({len(procedural_topics)}) - skip these")

# Create clean list for UI
ui_ready_topics = [row['subtopic'].title() for row in substantive_topics[:12]]
print(f"\n💎 UI-Ready Topics (Title Case): {ui_ready_topics}")

print(f"\n⚡ Quick decision: Use these {len(ui_ready_topics)} topics for the UI?")

In [ ]:
# Create Master Topic List - All 40 Substantive Topics
print("📋 Creating master topic list for UI...")

# Extract all substantive topics in title case
all_substantive_topics = [row['subtopic'].title() for row in substantive_topics]

# Create stats lookup for each topic
topic_stats_lookup = {}
for row in substantive_topics:
    topic_stats_lookup[row['subtopic'].title()] = {
        'responses': row['response_count'],
        'episodes': row['episode_count'], 
        'panelists': row['panelist_count'],
        'original': row['subtopic']  # Keep original for database queries
    }

print(f"✅ Master list created: {len(all_substantive_topics)} substantive topics")

# Create UI-friendly groupings
popular_topics = all_substantive_topics[:8]  # Top 8 for main chips
secondary_topics = all_substantive_topics[8:16]  # Next 8 for "more" section  
remaining_topics = all_substantive_topics[16:]  # Rest for full list

print(f"\n🎯 UI Strategy:")
print(f"Popular chips (always visible): {popular_topics}")
print(f"Secondary (show more): {secondary_topics}")
print(f"Remaining (full list): {len(remaining_topics)} topics")

# Ready for UI implementation
print(f"\n💎 Variables ready for create_new_ui():")
print(f"- all_substantive_topics: {len(all_substantive_topics)} topics")
print(f"- popular_topics: {len(popular_topics)} topics") 
print(f"- topic_stats_lookup: Stats for each topic")

print(f"\n⚡ Ready to implement topic chips UI! Time to build!")

In [ ]:
v

In [ ]:
def create_current_ui(helpers, qa_chain, config):
    """Create the current working UI - preserved as fallback"""
    
    with gr.Blocks(
        title="Q&A System V2 - Current UI",
        theme=gr.themes.Soft(),
        css="""
        .gr-radio-group {
            max-height: 200px;
            overflow-y: auto;
            border: 1px solid #e5e7eb;
            border-radius: 8px;
            padding: 12px;
            background: #fafafa;
        }
        .gr-radio-group label {
            padding: 4px 8px !important;
            margin: 2px 0 !important;
            border-radius: 4px;
        }
        .gr-radio-group label:hover {
            background: #f0f0f0;
        }
        #sample-questions {
            max-height: 200px !important;
            overflow-y: auto !important;
            border: 1px solid #ddd;
            border-radius: 6px;
            padding: 8px;
        }
        #question-container {
            position: relative !important;
        }
        #clear-btn-inside {
            position: absolute !important;
            top: 32px !important;
            right: 8px !important;
            width: 20px !important;
            height: 20px !important;
            min-width: 20px !important;
            padding: 0 !important;
            font-size: 12px !important;
            opacity: 0.5 !important;
            z-index: 10 !important;
            border-radius: 50% !important;
        }
        #clear-btn-inside:hover {
            opacity: 0.8 !important;
        }
        """    
    ) as demo:
        
        gr.Markdown("# 🧠 Q&A System V2 - Current UI")
        gr.Markdown("Current working interface with episode/panelist/topic filtering.")
        
        with gr.Row():
            # Left column - Filtering
            with gr.Column(scale=1):
                gr.Markdown("### 1️⃣ Select Content")
                
                episode_dropdown = gr.Dropdown(
                    label="Episode",
                    choices=sorted(helpers.episode_lookup.keys()),
                    value=sorted(helpers.episode_lookup.keys())[0],
                    interactive=True,
                )
                
                default_episode = sorted(helpers.episode_lookup.keys())[0]
                gr.Markdown("**Filter by:** *(based on selected episode)*")

                with gr.Row():
                    with gr.Column():
                        panellist_radio = gr.Radio(
                            label="Panellist (in this episode)", 
                            choices=helpers.get_panellists_by_episode(default_episode),
                            interactive=True
                        )
                    with gr.Column():
                        subtopic_radio = gr.Radio(
                            label="Topic (in this episode)", 
                            choices=helpers.get_subtopics_by_episode(default_episode),
                            interactive=True
                        )
            
            # Right column - Question input
            with gr.Column(scale=2):
                # Sample questions
                sample_questions = gr.Radio(
                    label="Sample Questions (click to select)",
                    choices=[
                        "What did panelists say about climate change?",
                        "How do panelists view immigration policy?", 
                        "What are different perspectives on the economy?",
                    ],
                    interactive=True,
                    elem_id="sample-questions"
                )   
                
                # Question input with clear button
                with gr.Row():
                    with gr.Column(scale=1, elem_id="question-container"):
                        question = gr.Textbox(
                            label="Your Question",
                            placeholder="e.g., What did panelists say about climate change?",
                            lines=3,
                            elem_id="question-input"
                        )
                        clear_question_btn = gr.Button(
                            "✕", 
                            variant="secondary", 
                            size="sm",
                            elem_id="clear-btn-inside"
                        )
                
                # Controls
                with gr.Row():
                    k_slider = gr.Slider(
                        minimum=5,
                        maximum=50, 
                        value=15,
                        step=5,
                        label="Documents to retrieve (k)"
                    )
                    
                    style_radio = gr.Radio(
                        label="Response Style",
                        choices=["Concise", "Balanced", "Detailed"],
                        value="Balanced"
                    )
                
                # Buttons
                with gr.Row():
                    submit_btn = gr.Button(
                        "🔍 Ask Question", 
                        variant="primary",
                        interactive=False
                    )   
                    clear_btn = gr.Button("🗑️ Clear", variant="secondary")
            
            with gr.Column(scale=1):
                # Status
                gr.Markdown("### System Status")
                status_display = gr.Textbox(
                    label="Status",
                    value="Ready to test!",
                    interactive=False
                )
                
                # Config info
                gr.Markdown(f"""
                **Configuration:**
                - Model: {config.chat_model_name}
                - Temperature: {config.temperature}
                - Debug: {config.debug_enabled}
                """)
        
        # Results
        gr.Markdown("### 📝 Response")
        answer_display = gr.Markdown(
            value="*Ask a question to see the response here.*"
        )
        
        gr.Markdown("### 📚 Sources") 
        sources_display = gr.Markdown(
            value="*Sources will appear here.*"
        )
        
        # Helper functions for this UI
        def set_sample_question(selected_question):
            return selected_question if selected_question else ""
        
        def clear_inputs():
            return "", "*Ask a question to see the response here.*", "Cleared - ready for next question"
        
        def update_panellists(ep_label):
            new_panelists = helpers.get_panellists_by_episode(ep_label)
            return gr.update(choices=new_panelists, value=None)

        def update_subtopics(ep_label):
            new_subtopics = helpers.get_subtopics_by_episode(ep_label)
            return gr.update(choices=new_subtopics, value=None)

        def filter_subtopics_by_panellist(panellist_name, ep_label):
            if panellist_name is None:
                return gr.update()
            new_subtopics = helpers.get_subtopics_by_panellist(ep_label, panellist_name)
            return gr.update(choices=new_subtopics, value=None)

        def clear_question():
            return "", gr.update(interactive=False)

        def update_submit_button(question_text):
            has_content = bool(question_text.strip())
            return gr.update(interactive=has_content)
        
        # Wire up events
        episode_dropdown.change(
            fn=update_panellists,
            inputs=episode_dropdown,
            outputs=panellist_radio,
        )
        episode_dropdown.change(
            fn=update_subtopics,
            inputs=episode_dropdown,
            outputs=subtopic_radio,
        )
        panellist_radio.change(
            fn=filter_subtopics_by_panellist,
            inputs=[panellist_radio, episode_dropdown],
            outputs=subtopic_radio,
        )
        sample_questions.change(
            fn=set_sample_question,
            inputs=[sample_questions],
            outputs=[question]
        )
        panellist_radio.change(
            fn=update_sample_questions,
            inputs=[panellist_radio, subtopic_radio],
            outputs=sample_questions
        )
        subtopic_radio.change(
            fn=update_sample_questions,
            inputs=[panellist_radio, subtopic_radio], 
            outputs=sample_questions
        )
        submit_btn.click(
            fn=handle_question,
            inputs=[question, k_slider, style_radio],
            outputs=[answer_display, sources_display, status_display]
        )
        clear_btn.click(
            fn=clear_inputs,
            inputs=[],
            outputs=[question, answer_display, status_display]
        )
        clear_question_btn.click(
            fn=clear_question,
            inputs=[],
            outputs=[question, submit_btn]
        )
        question.change(
            fn=update_submit_button,
            inputs=[question],
            outputs=[submit_btn]
        )
    
    return demo

print("✅ create_current_ui() function defined")